# 第18章 計算速度・精度・検証

書籍『Sionna RTによる電波伝搬レイトレーシング入門』第18章のコードである。
セルは本文の掲載順に並べてあり、コードセルの中身は本文に印刷したものと同じである。

先頭の「準備」セルだけは本文に印刷していない。本文のコードが前提にしている
import とシーンの読み込みをここで済ませてある。準備セルを実行すれば、以降は
上から順に実行できる。

ただし、本文が説明のために示している断片（自分で作ったシーンのパスや、その
シーンには無い物体名を使う例）はそのままでは動かない。該当するセルの前に注記を
置いてある。

## 準備

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# 本書の多くの章で使うミュンヘン中心部の都市シーン
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

## 2. 最大相互作用回数の影響

In [ ]:
import numpy as np
from sionna.rt import PathSolver

solver = PathSolver()
for depth in [0, 1, 3]:
    paths = solver(scene, max_depth=depth, los=True,
                   specular_reflection=True, seed=42)
    a = np.array(paths.a[0]) + 1j*np.array(paths.a[1])
    print(depth, int(np.array(paths.valid).sum()),
          10*np.log10(np.sum(np.abs(a)**2)))

## 3. 送信サンプル数の影響

In [ ]:
import numpy as np
from sionna.rt import RadioMapSolver

rms = RadioMapSolver()
for n in [10**4, 10**5, 10**6]:
    rm = rms(scene, max_depth=5, cell_size=[5.0, 5.0],
             samples_per_tx=n, seed=42)
    pg = np.array(rm.path_gain)[0]
    print(n, '未到達セル率 %.3f' % np.mean(pg == 0))

## 4. セルサイズの影響

In [ ]:
for cell in [1.0, 3.0, 10.0]:
    rm = rms(scene, max_depth=3, cell_size=[cell, cell],
             samples_per_tx=10**5, seed=42)
    print(cell, np.array(rm.path_gain).shape)   # (Tx, y, x)

```bash
# 計算を走らせている間、別の端末で使用量を記録する
nvidia-smi --query-gpu=memory.used --format=csv,noheader -l 1 \
  | tee gpu_mem.log
```

## 7. 自由空間損失との比較

In [ ]:
from sionna.rt import (load_scene, PlanarArray, Transmitter,
                       Receiver, PathSolver)

scene = load_scene()          # 引数なし = 物体のない空シーン
scene.frequency = 3.5e9
ant = PlanarArray(num_rows=1, num_cols=1,
                  pattern='iso', polarization='V')
scene.tx_array = ant
scene.rx_array = ant
scene.add(Transmitter('tx', position=[0.0, 0.0, 30.0]))
scene.add(Receiver('rx', position=[100.0, 0.0, 30.0]))

In [ ]:
import numpy as np

paths = PathSolver()(scene, max_depth=0, los=True,
                     specular_reflection=False)
a = np.array(paths.a[0]) + 1j*np.array(paths.a[1])
print('Sionna RT:', 10*np.log10(np.sum(np.abs(a)**2)))
print('theory   :', -(20*np.log10(100.0)
                      + 20*np.log10(3.5e9) - 147.55))

## 8. 2-ray モデルとの比較

In [ ]:
from sionna.rt import load_scene_from_string

GROUND_XML = """<scene version="2.1.0">
  <bsdf type="itu-radio-material" id="mat">
    <string name="type" value="concrete"/>
  </bsdf>
  <shape type="rectangle" id="ground">
    <transform name="to_world"><scale value="3000"/></transform>
    <ref id="mat" name="bsdf"/>
  </shape>
</scene>"""

scene = load_scene_from_string(GROUND_XML)   # 6 km 四方の地面だけ
scene.frequency = 3.5e9

In [ ]:
paths = PathSolver()(scene, max_depth=1, los=True,
                     specular_reflection=True, seed=42)
a = np.array(paths.a[0]) + 1j*np.array(paths.a[1])
print('paths:', int(np.array(paths.valid).sum()))   # 2 本になる
print('gain [dB]:', 10*np.log10(np.sum(np.abs(a)**2)))